# task_10 Solution

In [1]:

from pathlib import Path
import pandas as pd

base = Path("../../tasks/task_10/data/")


In [2]:

depots = pd.read_csv(base / "depots.csv")
scans = pd.read_csv(base / "scans.csv")
shipments = pd.read_csv(base / "shipments.csv")

Q1: For shipments where in_transit scan count == promised_days, compute delivery hours (picked_up → delivered). Which carrier has the lowest median?

In [3]:
shipments["ship_date"] = pd.to_datetime(shipments["ship_date"])
scans["scan_time"] = pd.to_datetime(scans["scan_time"])

first_events = scans.sort_values("scan_time").groupby(["shipment_id", "status"])["scan_time"].first().unstack()
ship = shipments.merge(first_events, on="shipment_id")

# Count in_transit scans per shipment
transit_counts = scans[scans["status"] == "in_transit"].groupby("shipment_id").size().reset_index(name="n_transit")
ship = ship.merge(transit_counts, on="shipment_id", how="left")

# Filter: n_transit == promised_days
matching = ship[ship["n_transit"] == ship["promised_days"]].copy()
matching["delivery_hours"] = (matching["delivered"] - matching["picked_up"]).dt.total_seconds() / 3600

q1 = matching.groupby("carrier")["delivery_hours"].median().sort_values().index[0]
print(q1)

CR1


Q2: For shipments originating from South region depots, what is the median number of hours between picking up and recieving at the hub?

In [4]:
south_depots = set(depots[depots["region"] == "South"]["depot_id"])
south_ship = ship[ship["origin_depot"].isin(south_depots)].copy()
south_ship["pickup_to_hub_hours"] = (south_ship["hub_received"] - south_ship["picked_up"]).dt.total_seconds() / 3600
q2 = str(south_ship["pickup_to_hub_hours"].median())
print(q2)

12.5


Q4: For each depot, find shipments above that depot's median weight. What % of those were delivered late?

In [5]:
# Delivery hours from picked_up to delivered
ship["delivery_hours"] = (ship["delivered"] - ship["picked_up"]).dt.total_seconds() / 3600
ship["late"] = ship["delivery_hours"] > (ship["promised_days"] * 24)

# Per-depot median weight
depot_median_weight = ship.groupby("origin_depot")["weight_kg"].median()

# For each depot, keep shipments above that depot's median
above_parts = []
for depot, med in depot_median_weight.items():
    above = ship[(ship["origin_depot"] == depot) & (ship["weight_kg"] > med)]
    above_parts.append(above)
above_depot_median = pd.concat(above_parts)

q4 = f"{above_depot_median['late'].mean() * 100:.1f}%"
print(q4)

50.0%
